In [ ]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np

In [ ]:
df=pd.read_csv(r"C:\Users\asuna\Downloads\address-data.csv")

In [ ]:
df.head

In [ ]:
df.nunique()

In [ ]:
df.info()

In [ ]:
list=df['Record Address'].unique()

In [ ]:
len(list)

In [ ]:
list

In [ ]:
model = SentenceTransformer('pawan2411/address-emnet')
em=model.encode(list,convert_to_tensor=True)

In [ ]:
em.shape

In [ ]:
threshold = 0.80  # cosine similarity threshold
remaining_indices = set(range(len(em)))
clusters = []

while remaining_indices:
    current_idx = next(iter(remaining_indices))

    sims = util.cos_sim(em[current_idx], em)[0]

    similar_idxs = torch.where(sims >= threshold)[0].tolist()

    similar_idxs = [i for i in similar_idxs if i in remaining_indices]

    cluster = [list[i] for i in similar_idxs]
    clusters.append(cluster)

    remaining_indices -= set(similar_idxs)

print(f"Formed {len(clusters)} clusters.\n")

for i, c in enumerate(clusters, 1):
    print(f"Cluster {i} ({len(c)} addresses):")
    for addr in c:
        print("   ", addr)
    print()


In [ ]:
clusters[1]

In [ ]:
data = []
for cluster in clusters:
    first = cluster[0]
    rest = cluster[1:] if len(cluster) > 1 else []
    count = len(rest)
    data.append({
        "original": first,
        "similar": rest,
        "count": count
    })

df = pd.DataFrame(data)
df['similar'] = df['similar'].astype(str).replace("[]", np.nan)

In [ ]:
df.head(10)

In [ ]:
output_path = r"C:\Users\asuna\Downloads\address-data.xlsx"
new_sheet_name = "Clustered_Addresses"

# Append DataFrame to a new sheet of the existing Excel file
with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='new') as writer:
    df.to_excel(writer, index=False, sheet_name=new_sheet_name)

print(f"✅ DataFrame successfully added as new sheet '{new_sheet_name}' in {output_path}")
